# Fine-tune Moirai-MoE-base v4 -- multi-index training data + macro covariates

v2 (full fine-tune) and v3 (freeze_ffn + covariates + real early stopping) converged to *nearly
identical*, zero-shot-losing results despite very different training recipes. That's the key clue:
the ceiling wasn't really about how conservatively we fine-tune, it was about what the model could
learn from what we fed it. v4 changes the data, not just the recipe:

1. **Fine-tune on 4 major US equity indices** (S&P 500, Nasdaq Composite, Dow, Russell 2000) instead
   of one. Every window in v2/v3 came from the *same* series just shifted in time -- a 935M-param
   model fine-tuned on that has no independent series to learn generalizable structure from, only one
   history to overfit. This required generalizing uni2ts's data builder to support multiple items
   (`--tickers`), not just per-item covariates (which v3 already added).
2. **Real exogenous covariates**: VIX (volatility) and the 10-year Treasury yield, shared across every
   index. Momentum/MovingAverage (kept from v3) are pure functions of Close -- the model can already
   derive them from its own context, so they add little. VIX/yield are information the model can't
   derive from price history alone.
3. **Lower learning rate** (1e-5 -> 3e-6) -- untested lever; both v2 and v3 used 1e-5 regardless of
   how many parameters were trainable.
4. **Evaluation methodology fixes**: point forecasts now use the sample **median** (not mean --
   consistent with what Lightning's own validation loop already uses for point metrics, and more
   robust to a few extreme sampled paths from a mixture-of-experts model); a **naive-persistence
   baseline** (predict "no change") is reported alongside zero-shot/fine-tuned so we can finally tell
   whether either model beats the trivial answer; test windows now **overlap** (denser sampling) for a
   statistically sturdier comparison than 14 non-overlapping windows.
5. **Still only backtested on S&P 500** -- the other 3 indices are training-diversity fuel, not part of
   what we're graded on.

**Requires a GPU runtime**: Runtime -> Change runtime type -> T4 GPU (or better).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: no GPU detected. Go to Runtime -> Change runtime type -> select a GPU, then re-run this cell.")

## 1. Clone the repo and install dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Agrim-Nuware/MOIRAI-CODE.git"
if not os.path.isdir("repo"):
    !git clone $REPO_URL repo
%cd repo

In [ ]:
!pip install -q -e '.[notebook]'
!pip install -q bitsandbytes yfinance

# uni2ts pins numpy~=1.26, which downgrades Colab's preinstalled numpy 2.x.
# Colab's preinstalled pandas is built against numpy 2.x, so once numpy is
# downgraded the two are ABI-incompatible ("numpy.dtype size changed").
# Force-reinstall a matching pandas, then restart the runtime so every
# already-imported module (numpy got pulled in transitively by torch above)
# reloads consistently. This cell intentionally crashes/restarts the kernel --
# that's expected, not an error. After it restarts, just continue running
# from the next cell (installed packages and cloned files are unaffected).
!pip install -q --force-reinstall "numpy<2" "pandas>=2.0,<2.3"
import os

os.kill(os.getpid(), 9)

**The cell above deliberately restarts the Colab runtime** (to fix a numpy/pandas
version mismatch). You'll see a "session crashed" / "automatically restarted" notice --
that's expected. Once it restarts, just continue running the cells below in order;
you do **not** need to re-run the clone or pip install cells.

In [ ]:
%cd /content/repo
import numpy as np
import pandas as pd

print("numpy:", np.__version__, "| pandas:", pd.__version__)

In [ ]:
with open(".env", "w") as f:
    f.write("CUSTOM_DATA_PATH=dataset/uni2ts_storage\n")
print(open(".env").read())

## 2. Download 4 equity indices + VIX/10y yield, engineer features, split at 2024-10-01

In [ ]:
!python dataset/sp500/prepare_v4_data.py

In [ ]:
import json
import pandas as pd

with open("dataset/sp500/split_info_v4.json") as f:
    split_info = json.load(f)

trainval_df = pd.read_csv("dataset/sp500/sp500_v4_model_input_trainval.csv", index_col=0, parse_dates=True)
train_length = split_info["train_length"]
date_offset = trainval_df.index[train_length - 1].strftime("%Y-%m-%d")
tickers = ",".join(split_info["tickers"])
target_columns = ",".join(split_info["target_suffixes"])
covariate_columns = ",".join(split_info["covariate_suffixes"])
shared_covariate_columns = ",".join(split_info["shared_covariate_columns"])

print("tickers (training diversity):", tickers)
print("eval ticker (only one backtested):", split_info["eval_ticker"])
print("per-ticker target columns:", target_columns)
print("per-ticker covariate columns:", covariate_columns)
print("shared covariate columns:", shared_covariate_columns)
print("train_length:", train_length)
print("lightning val offset:", split_info["lightning_val_offset"])
print("lightning val length:", split_info["lightning_val_length"])
print("final test length (>= cutoff):", split_info["final_test_len"])
print("date_offset for CSV builder:", date_offset)

## 3. Build the uni2ts HF-format dataset (4 items, target + covariate split)

In [ ]:
!python -m uni2ts.data.builder.simple SP500V4 dataset/sp500/sp500_v4_model_input_trainval.csv \
  --dataset_type wide_multivariate_covariates \
  --tickers "{tickers}" \
  --target_columns "{target_columns}" \
  --covariate_columns "{covariate_columns}" \
  --shared_covariate_columns "{shared_covariate_columns}" \
  --date_offset "{date_offset}" --freq B

## 4. Fine-tune Moirai-MoE-base (freeze_ffn, lower LR, GPU, fp16 + 8-bit AdamW, real early stopping)

`data.mode=MC` / `val_data.mode=MC` route through the multi-item covariate-aware dataset builder --
each of the 4 indices contributes its own sliding windows every epoch, and Lightning validation now
spans all 4 indices' held-back regions too (a less narrow, more robust signal for picking the best
checkpoint than v2/v3's S&P-500-only validation). `model.lr=3e-6` is a new, lower learning rate --
both v2 (full) and v3 (freeze_ffn) used 1e-5 regardless of trainable-parameter count, so this is the
one untested recipe lever. No `trainer.callbacks.2.patience` override -- default `patience=3` applies.

In [ ]:
lightning_val_offset = split_info["lightning_val_offset"]
lightning_val_length = split_info["lightning_val_length"]

!python -m cli.train \
  -cp conf/finetune \
  exp_name=sp500_v4_full_finetune \
  run_name=run1 \
  tf32=false \
  model=moirai_moe_1.0_R_base \
  model.patch_size=16 \
  model.context_length=512 \
  model.prediction_length=32 \
  model.num_training_steps=3000 \
  model.num_warmup_steps=50 \
  model.finetune_pattern=freeze_ffn \
  model.use_8bit_adam=true \
  model.lr=3e-6 \
  data=sp500 \
  data.dataset=SP500V4 \
  data.patch_size=16 \
  data.context_length=512 \
  data.prediction_length=32 \
  data.mode=MC \
  data.train_length={train_length} \
  data.distance=16 \
  val_data=sp500 \
  val_data.dataset=SP500V4_eval \
  val_data.patch_size=16 \
  val_data.context_length=512 \
  val_data.prediction_length=32 \
  val_data.mode=MC \
  val_data.offset={lightning_val_offset} \
  val_data.eval_length={lightning_val_length} \
  val_data.distance=32 \
  trainer.max_epochs=30 \
  trainer.accelerator=gpu \
  trainer.devices=1 \
  trainer.precision=16-mixed \
  +trainer.log_every_n_steps=10 \
  train_dataloader.batch_size=4 \
  train_dataloader.num_workers=0 \
  val_dataloader.batch_size=2 \
  val_dataloader.num_workers=0

## 5. Evaluate: zero-shot vs fine-tuned vs naive persistence, on S&P 500's held-out region (>= 2024-10-01)

In [ ]:
!python dataset/sp500/evaluate_finetuned_v4.py \
  --context_length 512 --prediction_length 32 --num_samples 150

In [ ]:
from IPython.display import Image, display

print("Forecast comparison (Close price -- direct vs reconstructed vs naive):")
display(Image("dataset/sp500/results_v4_forecast_plot.png"))
print("\nError comparison by variate:")
display(Image("dataset/sp500/results_v4_metrics_bar.png"))
print("\nFine-tuning loss curve:")
display(Image("dataset/sp500/results_v4_loss_curve.png"))

In [ ]:
import json

with open("dataset/sp500/results_v4_metrics.json") as f:
    results = json.load(f)

naive, zs, ft = results["naive"], results["zero_shot"], results["fine_tuned"]
print(f"{'metric':<22} {'naive':>12} {'zero-shot':>12} {'fine-tuned':>12}")
print(f"{'Close MAPE':<22} {naive['Close']['mape']:>11.2f}% {zs['Close']['mape']:>11.2f}% {ft['Close']['mape']:>11.2f}%")
print(f"{'Close(recon) MAPE':<22} {naive['Close_reconstructed']['mape']:>11.2f}% {zs['Close_reconstructed']['mape']:>11.2f}% {ft['Close_reconstructed']['mape']:>11.2f}%")
print(f"{'Return MAE':<22} {naive['Return']['mae']:>12.5f} {zs['Return']['mae']:>12.5f} {ft['Return']['mae']:>12.5f}")
print(f"{'LogReturn MAE':<22} {naive['LogReturn']['mae']:>12.5f} {zs['LogReturn']['mae']:>12.5f} {ft['LogReturn']['mae']:>12.5f}")

## 6. (Optional) Save results back to your GitHub repo

Uncomment and fill in a [personal access token](https://github.com/settings/tokens) if you
want to push the plots/metrics back to your repo. Skip this if you'd rather just download
the files from the Colab file browser (left sidebar).

In [ ]:
# GITHUB_TOKEN = ""  # paste a token with repo write access, or leave blank to skip
# if GITHUB_TOKEN:
#     !git add dataset/sp500/results_v4_*.png dataset/sp500/results_v4_metrics.json
#     !git commit -m "Add Colab v4 fine-tuning results"
#     !git push https://$GITHUB_TOKEN@github.com/Agrim-Nuware/MOIRAI-CODE.git HEAD:main